In [ ]:
%run  Data_preparation_TCGA.ipynb
## More information about the dataset can be found in Data_preparation_TCGA.ipynb

In [ ]:
from codes.utils.util import *
from codes.drugcell_NN import *

training_file = "data/drugcell_train.txt"
testing_file = "data/drugcell_test.txt"
val_file = "data/drugcell_val.txt"
cell2id_file = "data/cell2ind.txt"
drug2id_file = "data/drug2ind.txt"
genotype_file = "data/cell2mutation.txt"
fingerprint_file = "data/drug2fingerprint.txt"
onto_file = "data/drugcell_ont.txt"
gene2id_file = "data/gene2ind.txt"


# load ontology
dG, root, term_size_map, \
    term_direct_gene_map = load_ontology(onto_file, 
                                         gene2id_mapping)

In [ ]:
import networkx as nx
import random
import numpy as np
from collections import defaultdict

def generate_maximally_different_dags(original_graph, tries=50, k=10, seed=None):
    
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    nodes = list(nx.topological_sort(original_graph))  # Topological order
    layer = {n: i for i, n in enumerate(nodes)}

    candidates = []

    for _ in range(tries):
        new_graph = nx.DiGraph()
        new_graph.add_nodes_from(original_graph.nodes)

        # Step 1: Group nodes by out-degree
        outdeg_buckets = defaultdict(list)
        for n in original_graph.nodes:
            outdeg_buckets[original_graph.out_degree(n)].append(n)

        # Step 2: Build new random neighbors (pointing only to later layers)
        for outdeg, bucket in outdeg_buckets.items():
            for node in bucket:
                candidate_nodes = [n for n in original_graph.nodes if layer[n] > layer[node]]

                if len(candidate_nodes) >= outdeg:
                    new_neighbors = random.sample(candidate_nodes, outdeg)
                else:
                    # If there are not enough candidates, use all of them
                    new_neighbors = candidate_nodes

                for v in new_neighbors:
                    new_graph.add_edge(node, v)

        # Step 3: Evaluate difference (Jaccard similarity)
        e1 = set(original_graph.edges())
        e2 = set(new_graph.edges())
        inter = len(e1 & e2)
        union = len(e1 | e2)
        jacc = inter / union if union > 0 else 0
        score = 1 - jacc

        candidates.append((score, new_graph))

    # Select the top k graphs with the highest difference score
    candidates.sort(key=lambda x: x[0], reverse=True)
    top_graphs = candidates[:k]

    # Check edge count consistency
    for _, g in top_graphs:
        assert g.number_of_edges() == original_graph.number_of_edges(), \
            f"Edge count mismatch: original {original_graph.number_of_edges()}, new {g.number_of_edges()}"

    return top_graphs



top10_graphs = generate_maximally_different_dags(dG, tries=100, k=10, seed=42)

print("Original edge count:", dG.number_of_edges())
for i, (score, g) in enumerate(top10_graphs, 1):
    print(f"Graph {i}: nodes={g.number_of_nodes()}, edges={g.number_of_edges()}, "
          f"difference={score:.4f}, is_DAG={nx.is_directed_acyclic_graph(g)}")


In [ ]:
import networkx as nx
import random
import numpy as np

def generate_random_dags(original_graph, k=10, tries=100, seed=None, ensure_unique=True):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # Topological order and layer mapping
    topo_nodes = list(nx.topological_sort(original_graph))
    layer = {n: i for i, n in enumerate(topo_nodes)}

    # Out-degree of each node in the original graph
    outdeg_map = {n: original_graph.out_degree(n) for n in original_graph.nodes}
    m = original_graph.number_of_edges()

    # Pre-check: each node must have enough candidate successors
    for u in original_graph.nodes:
        candidates = [v for v in original_graph.nodes if layer[v] > layer[u]]
        if len(candidates) < outdeg_map[u]:
            raise ValueError("Invalid graph: insufficient candidate successors for some nodes.")

    results = []
    seen = set()  # Used for deduplication (store edge sets as frozensets)

    attempts = 0
    while len(results) < k and attempts < tries:
        attempts += 1

        g = nx.DiGraph()
        g.add_nodes_from(original_graph.nodes)

        # For each node, randomly select the same number of successors as in the original graph
        for u in topo_nodes:
            outdeg = outdeg_map[u]
            candidates = [v for v in original_graph.nodes if layer[v] > layer[u]]
            successors = random.sample(candidates, outdeg)
            g.add_edges_from((u, v) for v in successors)

        # Ensure edge count consistency
        if g.number_of_edges() != m:
            continue

        if ensure_unique:
            key = frozenset(g.edges())
            if key in seen:
                continue
            seen.add(key)

        results.append(g)

    return results


rand_graphs = generate_random_dags(dG, k=10, tries=200, seed=42, ensure_unique=True)

print("Original: nodes =", dG.number_of_nodes(), "edges =", dG.number_of_edges())
for i, g in enumerate(rand_graphs, 1):
    print(f"Graph {i}: nodes={g.number_of_nodes()}, edges={g.number_of_edges()}, is_DAG={nx.is_directed_acyclic_graph(g)}")
